# QuantJourney SDK - Options, VIX, SKEW and Term Structure

This notebook demonstrates a QuantJourney SDK workflow that inspects volatility feeds and options context around VIX, VVIX, SKEW, term structure and option-chain metadata.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


In [ ]:
symbol = 'SPY'
vix_raw = qj.cboe.get_vix_data(start_date='2020-01-01', end_date=END)
vvix_raw = qj.cboe.get_vvix_data(start_date='2020-01-01', end_date=END)
skew_raw = qj.cboe.get_skew_index_data(start_date='2020-01-01', end_date=END)
term_raw = qj.cboe.get_vix_term_structure()
expirations_raw = qj.cboe.get_options_expirations(symbol=symbol)
chain_raw = qj.cboe.get_options_chain(symbol=symbol)


In [ ]:
def series_from_rows(payload: Any, name: str) -> pd.Series:
    rows = pd.DataFrame(as_rows(payload))
    if rows.empty:
        return pd.Series(dtype=float, name=name)
    date_col = next((col for col in rows.columns if 'date' in str(col).lower()), rows.columns[0])
    numeric_cols = rows.select_dtypes(include='number').columns.tolist()
    value_col = 'close' if 'close' in rows.columns else numeric_cols[-1] if numeric_cols else None
    if value_col is None:
        return pd.Series(dtype=float, name=name)
    rows['date'] = pd.to_datetime(rows[date_col], errors='coerce')
    rows[name] = pd.to_numeric(rows[value_col], errors='coerce')
    return rows.dropna(subset=['date', name]).set_index('date')[name].sort_index()
vol = pd.concat([series_from_rows(vix_raw, 'vix'), series_from_rows(vvix_raw, 'vvix'), series_from_rows(skew_raw, 'skew')], axis=1).dropna(how='all')
feed_summary = pd.Series({'vix_rows': len(as_rows(vix_raw)), 'vvix_rows': len(as_rows(vvix_raw)), 'skew_rows': len(as_rows(skew_raw)), 'term_structure_rows': len(as_rows(term_raw)), 'expiration_rows': len(as_rows(expirations_raw)), 'chain_rows': len(as_rows(chain_raw))})
display(feed_summary)


In [ ]:
if not vol.empty:
    vol.tail(1000).plot(title='Volatility feed context')
    plt.ylabel('index level')
    plt.show()
    latest = vol.tail(252).describe().T
    display(latest)


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.